# Data Preparation & Feature Engineering
- This notebook prepares the DHS cross-sectional and MAL-ED longitudinal datasets.
- It applies WHO labeling rules and prepares the feature arrays for the ML models.

## Initial Setup and Imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

## Load Data and Inspect (Both Datasets)

In [2]:
# Load the datasets
dhs_df = pd.read_csv('../data/synthetic_data/dhs_2016_raw_synthetic.csv')
maled_df = pd.read_csv('../data/synthetic_data/mal-ed_longitudinal_raw_synthetic.csv')

def inspect_dataset(df, name):
    # Utility function to print shape, missing values, and dtypes.
    print(f"--- {name} ---")
    print(f"Shape: {df.shape}")
    
    missing = df.isnull().sum()
    missing_count = missing[missing > 0]
    print("\nMissing Values:")
    print(missing_count if not missing_count.empty else "No missing values detected.")
    
    print("\nData Types:")
    print(df.dtypes)
    print("=" * 50, "\n")

# Inspect both datasets
inspect_dataset(dhs_df, "DHS Dataset (Shallow Learning)")
inspect_dataset(maled_df, "MAL-ED Dataset (Longitudinal Deep Learning)")

--- DHS Dataset (Shallow Learning) ---
Shape: (5000, 16)

Missing Values:
birth_weight_kg                      464
birth_length_cm                      479
vaccination_adherence_score          319
missed_clinic_count_last_6_months    463
dtype: int64

Data Types:
child_id                              object
date_of_birth                         object
date_of_measurement                   object
age_in_days                            int64
sex                                    int64
weight_kg                            float64
height_cm                            float64
MUAC_mm                              float64
oedema                                 int64
birth_weight_kg                      float64
birth_length_cm                      float64
birth_order                            int64
vaccination_adherence_score          float64
missed_clinic_count_last_6_months    float64
breastfeeding_status                  object
sector                                object
dtype: object

-

## Basic Standardization and Sorting

In [3]:
# ---------------------------------------------------------
# DHS Step 2: Standardize sex column -> enforce strictly 1 / 2
# ---------------------------------------------------------
print("DHS 'sex' values before standardizing:", dhs_df['sex'].unique())

# Robust mapping dictionary to catch string variations in synthetic data
# Assuming standard demographic coding: 1 = Male, 0 = Female
sex_mapping = {
    1: 1,
    2: 0
}

# Clean, map, and coerce
dhs_df['sex'] = dhs_df['sex'].map(sex_mapping)
print("DHS 'sex' values after standardizing:", dhs_df['sex'].unique())

# ---------------------------------------------------------
# MAL-ED Step 2: Sort by child_id -> visit_number ascending
# ---------------------------------------------------------
maled_df = maled_df.sort_values(by=['child_id', 'visit_number'], ascending=[True, True])
maled_df = maled_df.reset_index(drop=True)

print("\nMAL-ED dataset successfully sorted chronologically by child_id and visit_number.")

DHS 'sex' values before standardizing: [1 2]
DHS 'sex' values after standardizing: [1 0]

MAL-ED dataset successfully sorted chronologically by child_id and visit_number.


## DHS — Date Parsing and Age Validation

In [4]:
# ---------------------------------------------------------
# DHS Step 3: Parse dates and validate age_in_days
# ---------------------------------------------------------
print("DHS Shape before age validation:", dhs_df.shape)

# Convert to datetime objects
dhs_df['date_of_birth'] = pd.to_datetime(dhs_df['date_of_birth'], errors='coerce')
dhs_df['date_of_measurement'] = pd.to_datetime(dhs_df['date_of_measurement'], errors='coerce')

# Calculate expected age
expected_age_days = (dhs_df['date_of_measurement'] - dhs_df['date_of_birth']).dt.days

# Fill missing age_in_days
dhs_df['age_in_days'] = dhs_df['age_in_days'].fillna(expected_age_days)

# Drop rows where dates are corrupted (negative age or missing critical dates)
dhs_df = dhs_df[(dhs_df['age_in_days'] >= 0) & (dhs_df['age_in_days'].notnull())]

print("DHS Shape after age validation:", dhs_df.shape)

DHS Shape before age validation: (5000, 16)
DHS Shape after age validation: (5000, 16)


## DHS - Outlier Removal

In [5]:
# ---------------------------------------------------------
# DHS Step 4: Remove biologically implausible outliers - WHO measures
# ---------------------------------------------------------
# Assuming children under 5 (0-59 months):
# Weight: 1kg to 35kg | Height: 40cm to 130cm | MUAC: 70mm to 250mm
valid_weight = (dhs_df['weight_kg'] >= 1.0) & (dhs_df['weight_kg'] <= 35.0)
valid_height = (dhs_df['height_cm'] >= 40.0) & (dhs_df['height_cm'] <= 130.0)

# MUAC might have NaNs (to be imputed later), so we only filter non-NaN outliers
valid_muac = dhs_df['MUAC_mm'].isna() | ((dhs_df['MUAC_mm'] >= 70) & (dhs_df['MUAC_mm'] <= 250))

dhs_df = dhs_df[valid_weight & valid_height & valid_muac]

print("DHS Shape after outlier removal:", dhs_df.shape)

DHS Shape after outlier removal: (5000, 16)


## MAL-ED - Monotonicity and Temporal Feature Engineering

In [6]:
# ---------------------------------------------------------
# MAL-ED Steps 3 & 4: Monotonicity checks & Temporal Engineering
# ---------------------------------------------------------
maled_df['visit_date'] = pd.to_datetime(maled_df['visit_date'], errors='coerce')
maled_df['date_of_birth'] = pd.to_datetime(maled_df['date_of_birth'], errors='coerce')

# Robust mapping dictionary to catch string variations in synthetic data
# Assuming standard demographic coding: 1 = Male, 0 = Female
sex_mapping = {
    1: 1,
    2: 0
}

# Clean, map, and coerce
maled_df['sex'] = maled_df['sex'].map(sex_mapping)

# Group by child_id for sequential operations
grouped_maled = maled_df.groupby('child_id')

# 1. Height Monotonicity: Fix clinical dips (a child's height shouldn't decrease)
maled_df['height_cm'] = grouped_maled['height_cm'].cummax()

# 2. Age Monotonicity: Ensure age doesn't go backwards
maled_df['age_in_days'] = grouped_maled['age_in_days'].cummax()

# 3. Engineer LSTM sequence features
maled_df['days_since_last_visit'] = grouped_maled['visit_date'].diff().dt.days
maled_df['weight_delta_kg'] = grouped_maled['weight_kg'].diff()

# 4. Handle the first visit in each sequence (diff returns NaN)
# We fill with 0 because, at step 1 of the sequence, there is no delta.
maled_df['days_since_last_visit'] = maled_df['days_since_last_visit'].fillna(0)
maled_df['weight_delta_kg'] = maled_df['weight_delta_kg'].fillna(0)

print("Engineered sequential features for MAL-ED:")
display(maled_df[['child_id', 'visit_number', 'date_of_birth', 'age_in_days', 'height_cm', 'days_since_last_visit', 'weight_delta_kg']].head(10))

Engineered sequential features for MAL-ED:


,child_id,visit_number,date_of_birth,age_in_days,height_cm,days_since_last_visit,weight_delta_kg
0,BCD-00001,1,2016-06-07,97,63.33,0.0,0.00
1,BCD-00001,2,2016-06-07,116,64.14,19.0,0.52
2,BCD-00001,3,2016-06-07,183,68.97,67.0,0.65
3,BCD-00001,4,2016-06-07,244,71.40,61.0,0.47
4,BCD-00002,1,2015-08-24,51,54.94,0.0,0.00
5,BCD-00002,2,2015-08-24,97,62.42,46.0,0.61
6,BCD-00002,3,2015-08-24,158,66.81,61.0,1.79
7,BCD-00002,4,2015-08-24,174,67.08,16.0,2.91
8,BCD-00002,5,2015-08-24,209,67.70,35.0,0.56
9,BCD-00003,1,2017-06-12,63,51.81,0.0,0.00


## WHO Z-Score Calculation (LMS Method)

In [7]:
# ---------------------------------------------------------
# Step 6: Real WHO LMS Z-Score Calculation (WFL / WFH)
# ---------------------------------------------------------
import os

# 1. Load the WHO reference tables from the specified folder
# (Make sure these filenames match exactly how you saved them)
wfl_boys = pd.read_csv('../data/who_references/wfl-boys-zscore-expanded-table.csv')
wfl_girls = pd.read_csv('../data/who_references/wfl-girls-zscore-expanded-table.csv')
wfh_boys = pd.read_csv('../data/who_references/wfh-boys-zscore-expanded-tables.csv')
wfh_girls = pd.read_csv('../data/who_references/wfh-girls-zscore-expanded-tables.csv')

# 2. Standardize the measurement column name to 'measure_cm' so they can be combined
wfl_boys = wfl_boys.rename(columns={'Length': 'measure_cm'})
wfl_girls = wfl_girls.rename(columns={'Length': 'measure_cm'})
wfh_boys = wfh_boys.rename(columns={'Height': 'measure_cm'})
wfh_girls = wfh_girls.rename(columns={'Height': 'measure_cm'})

# 3. Add identifiers so we can filter easily during the merge (Sex: 1=Male, 2=Female)
wfl_boys['sex'] = 1; wfl_boys['type'] = 'length'
wfl_girls['sex'] = 0; wfl_girls['type'] = 'length'
wfh_boys['sex'] = 1; wfh_boys['type'] = 'height'
wfh_girls['sex'] = 0; wfh_girls['type'] = 'height'

# 4. Combine all reference tables into one master lookup dataframe
who_master = pd.concat([wfl_boys, wfl_girls, wfh_boys, wfh_girls], ignore_index=True)

# Ensure the lookup measurement is cleanly rounded to 1 decimal place for the join
who_master['measure_cm'] = who_master['measure_cm'].round(1)

def calculate_exact_whz(df, master_lookup):
    """Calculates WHZ using the exact WHO master tables."""
    df_calc = df.copy()
    
    # WHO Rule: Length for < 730 days (< 2 years), Height for >= 730 days
    df_calc['type'] = np.where(df_calc['age_in_days'] < 730, 'length', 'height')
    
    # Round height in your dataset to the nearest 0.1cm to match WHO tables
    df_calc['measure_cm'] = df_calc['height_cm'].round(1)
    
    # Merge with the WHO master lookup table
    # We join on sex, measurement type (length/height), and the cm value
    df_calc = df_calc.merge(
        master_lookup[['sex', 'type', 'measure_cm', 'L', 'M', 'S']],
        on=['sex', 'type', 'measure_cm'],
        how='left'
    )
    
    # Extract merged values
    y = df_calc['weight_kg']
    L = df_calc['L']
    M = df_calc['M']
    S = df_calc['S']
    
    # Calculate Z-score using the LMS formula
    # Handle the L=0 edge case (uses natural log) just in case, though rare in WHO tables
    df_calc['WHZ'] = np.where(
        L == 0,
        np.log(y / M) / S,
        (((y / M) ** L) - 1) / (L * S)
    )
    
    return df_calc['WHZ']

# 5. Apply the calculation to both datasets
dhs_df['WHZ'] = calculate_exact_whz(dhs_df, who_master)
maled_df['WHZ'] = calculate_exact_whz(maled_df, who_master)

# 6. Sanity Check: Look for unmapped values 
# (This happens if a child's height is extremely out of bounds of the WHO tables)
print("DHS unmapped WHZ records:", dhs_df['WHZ'].isnull().sum())
print("MAL-ED unmapped WHZ records:", maled_df['WHZ'].isnull().sum())

# Optional: Drop rows that fell completely outside WHO theoretical bounds
dhs_df = dhs_df.dropna(subset=['WHZ'])
maled_df = maled_df.dropna(subset=['WHZ'])

print("\nExact WHO Z-scores calculated successfully.")

DHS unmapped WHZ records: 5
MAL-ED unmapped WHZ records: 0

Exact WHO Z-scores calculated successfully.


## WHO WAZ and HAZ Calculation (DHS Only)

In [8]:
# ---------------------------------------------------------
# Step 7: Real WHO LMS Calculation for WAZ and HAZ
# ---------------------------------------------------------

# 1. Load the WHO age-based reference tables (assuming similar naming convention)
# Weight-for-Age (WAZ)
wfa_boys = pd.read_csv('../data/who_references/wfa-boys-zscore-expanded-tables.csv')
wfa_girls = pd.read_csv('../data/who_references/wfa-girls-zscore-expanded-tables.csv')

# Height/Length-for-Age (HAZ)
hfa_boys = pd.read_csv('../data/who_references/lhfa-boys-zscore-expanded-tables.csv')
hfa_girls = pd.read_csv('../data/who_references/lhfa-girls-zscore-expanded-tables.csv')

# 2. Add sex identifiers
wfa_boys['sex'] = 1; wfa_girls['sex'] = 0
hfa_boys['sex'] = 1; hfa_girls['sex'] = 0

# 3. Combine into master lookups
wfa_master = pd.concat([wfa_boys, wfa_girls], ignore_index=True)
hfa_master = pd.concat([hfa_boys, hfa_girls], ignore_index=True)

def calculate_age_based_zscore(df, master_lookup, measure_col):
    """Calculates age-based Z-scores (WAZ, HAZ) using exact WHO tables."""
    df_calc = df.copy()
    
    # Merge on sex and age_in_days (WHO tables usually call this 'Day')
    # If your WHO csv has 'Age' instead of 'Day', change 'Day' below to match your column name.
    df_calc = df_calc.merge(
        master_lookup[['sex', 'Day', 'L', 'M', 'S']],
        left_on=['sex', 'age_in_days'],
        right_on=['sex', 'Day'],
        how='left'
    )
    
    y = df_calc[measure_col]
    L = df_calc['L']
    M = df_calc['M']
    S = df_calc['S']
    
    # LMS Calculation
    z_score = np.where(
        L == 0,
        np.log(y / M) / S,
        (((y / M) ** L) - 1) / (L * S)
    )
    
    return z_score

# 4. Apply to DHS dataset
# Calculate Weight-for-Age (WAZ)
dhs_df['WAZ'] = calculate_age_based_zscore(dhs_df, wfa_master, 'weight_kg')

# Calculate Height-for-Age (HAZ)
dhs_df['HAZ'] = calculate_age_based_zscore(dhs_df, hfa_master, 'height_cm')

# 5. Sanity Check
print("DHS unmapped WAZ records:", dhs_df['WAZ'].isnull().sum())
print("DHS unmapped HAZ records:", dhs_df['HAZ'].isnull().sum())

# Clean up any records that fell outside the 1856-day (5 year) WHO table limits
dhs_df = dhs_df.dropna(subset=['WAZ', 'HAZ'])

print("\nExact WHO WAZ and HAZ calculated successfully for DHS dataset.")

DHS unmapped WAZ records: 0
DHS unmapped HAZ records: 0

Exact WHO WAZ and HAZ calculated successfully for DHS dataset.


## Strict WHO Labeling

In [9]:
# ---------------------------------------------------------
# Step 8: Derive Target Labels (Normal, MAM, SAM)
# ---------------------------------------------------------
def apply_who_labels(row):
    """
    Applies WHO deterministic rules to classify malnutrition risk.
    0 = Normal, 1 = MAM, 2 = SAM
    """
    # 1. Oedema overrides all Z-scores -> always SAM
    # Strictly checking for numerical 1 as defined
    if pd.notna(row.get('oedema')) and row['oedema'] == 1:
        return 2 
    
    whz = row.get('WHZ', 0)
    muac = row.get('MUAC_mm', 999) # Default high if missing to not falsely trigger SAM/MAM
    
    # 2. SAM conditions
    if whz < -3.0 or muac < 115.0:
        return 2
        
    # 3. MAM conditions
    if (-3.0 <= whz < -2.0) or (115.0 <= muac < 125.0):
        return 1
        
    # 4. Normal
    return 0

# Apply to both datasets
dhs_df['label'] = dhs_df.apply(apply_who_labels, axis=1)
maled_df['label'] = maled_df.apply(apply_who_labels, axis=1)

print("--- DHS Label Distribution ---")
print(dhs_df['label'].value_counts(normalize=True) * 100)

print("\n--- MAL-ED Label Distribution ---")
print(maled_df['label'].value_counts(normalize=True) * 100)

--- DHS Label Distribution ---
label
0    76.756757
2    13.413413
1     9.829830
Name: proportion, dtype: float64

--- MAL-ED Label Distribution ---
label
0    55.166982
2    25.362319
1    19.470699
Name: proportion, dtype: float64


## Final Pre-Split Engineering

In [10]:
# ---------------------------------------------------------
# Step 9: MAL-ED Velocity & DHS Categorical Encoding
# ---------------------------------------------------------

# --- MAL-ED: Calculate WHZ Velocity ---
# Group by child_id to ensure we only diff within the same child's sequence
maled_df['whz_velocity'] = maled_df.groupby('child_id')['WHZ'].diff()

# Fill the first visit velocity with 0 (no prior data to compare against)
maled_df['whz_velocity'] = maled_df['whz_velocity'].fillna(0)

print("MAL-ED: whz_velocity engineered successfully.")

# --- DHS: Categorical Encoding ---
# Assuming 'breastfeeding_status' and 'sector' are the primary categoricals
categorical_cols = ['breastfeeding_status', 'sector']

# One-hot encode using get_dummies, drop_first=True prevents multicollinearity
dhs_df = pd.get_dummies(dhs_df, columns=[col for col in categorical_cols if col in dhs_df.columns], drop_first=True)

# Ensure all boolean outputs are converted to standard integers (0/1) for scikit-learn
for col in dhs_df.columns:
    if dhs_df[col].dtype == 'bool':
        dhs_df[col] = dhs_df[col].astype(int)

print("DHS: Categorical features encoded successfully.")

MAL-ED: whz_velocity engineered successfully.
DHS: Categorical features encoded successfully.


## Train/Test Split by `child_id`
By isolating unique `child_id`s first, we guarantee that a single child's records are exclusively in the training set or exclusively in the test set.

In [11]:
# ---------------------------------------------------------
# Step 10: Train / Test Split by Child ID (Prevent Data Leakage)
# ---------------------------------------------------------
from sklearn.model_selection import train_test_split

TEST_SIZE = 0.20
RANDOM_STATE = 42

def split_by_child_id(df):
    """Splits dataframe ensuring child_ids do not cross train/test boundaries."""
    unique_ids = df['child_id'].unique()
    
    # Split the IDs, not the rows
    train_ids, test_ids = train_test_split(unique_ids, test_size=TEST_SIZE, random_state=RANDOM_STATE)
    
    # Map the split IDs back to the original dataframe
    train_df = df[df['child_id'].isin(train_ids)].copy()
    test_df = df[df['child_id'].isin(test_ids)].copy()
    
    return train_df, test_df

# Execute splits
dhs_train, dhs_test = split_by_child_id(dhs_df)
maled_train, maled_test = split_by_child_id(maled_df)

print("--- DHS Split Results ---")
print(f"Train rows: {len(dhs_train)} | Test rows: {len(dhs_test)}")
print(f"Train distinct children: {dhs_train['child_id'].nunique()} | Test distinct children: {dhs_test['child_id'].nunique()}")

print("\n--- MAL-ED Split Results ---")
print(f"Train rows: {len(maled_train)} | Test rows: {len(maled_test)}")
print(f"Train distinct children: {maled_train['child_id'].nunique()} | Test distinct children: {maled_test['child_id'].nunique()}")

--- DHS Split Results ---
Train rows: 3996 | Test rows: 999
Train distinct children: 3996 | Test distinct children: 999

--- MAL-ED Split Results ---
Train rows: 2543 | Test rows: 631
Train distinct children: 560 | Test distinct children: 140


## DHS — MICE Imputation

In [12]:
# ---------------------------------------------------------
# DHS Step 10: MICE Imputation (Fit on Train, Transform Both)
# ---------------------------------------------------------
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

print("DHS Missing values before imputation (Train):", dhs_train.isnull().sum().sum())

# 1. Isolate continuous numerical columns for MICE to use for correlations
num_cols = dhs_train.select_dtypes(include=['float64', 'int64', 'int32']).columns.tolist()

# 2. Exclude identifiers and the target label to prevent data leakage!
exclude_cols = ['child_id', 'label']
mice_features = [col for col in num_cols if col not in exclude_cols]

# 3. Initialize MICE Imputer
mice_imputer = IterativeImputer(random_state=42, max_iter=10)

# 4. Fit strictly on the training set
mice_imputer.fit(dhs_train[mice_features])

# 5. Transform both Train and Test sets
dhs_train.loc[:, mice_features] = mice_imputer.transform(dhs_train[mice_features])
dhs_test.loc[:, mice_features] = mice_imputer.transform(dhs_test[mice_features])

print("DHS Missing values after MICE imputation (Train):", dhs_train.isnull().sum().sum())
print("DHS Missing values after MICE imputation (Test):", dhs_test.isnull().sum().sum())

DHS Missing values before imputation (Train): 1369
DHS Missing values after MICE imputation (Train): 0
DHS Missing values after MICE imputation (Test): 0


## MAL-ED - — Longitudinal Imputation

In [13]:
# ---------------------------------------------------------
# MAL-ED Steps 12 & 13: Longitudinal Imputation
# ---------------------------------------------------------

print("MAL-ED Missing before imputation (Train):", maled_train.isnull().sum().sum())

# Step 12: Forward-fill MUAC_mm per child 
# (Done separately on train and test to prevent cross-contamination)
maled_train['MUAC_mm'] = maled_train.groupby('child_id')['MUAC_mm'].ffill()
maled_test['MUAC_mm'] = maled_test.groupby('child_id')['MUAC_mm'].ffill()

# Step 13: Calculate medians strictly from the Training Set
train_medians = maled_train[['MUAC_mm', 'birth_weight_kg', 'birth_length_cm']].median()

print("\nCalculated Training Medians:")
print(train_medians)

# Apply these training medians to fill any remaining NaNs in both sets
maled_train = maled_train.fillna({
    'MUAC_mm': train_medians['MUAC_mm'],
    'birth_weight_kg': train_medians['birth_weight_kg'],
    'birth_length_cm': train_medians['birth_length_cm']
})

maled_test = maled_test.fillna({
    'MUAC_mm': train_medians['MUAC_mm'],
    'birth_weight_kg': train_medians['birth_weight_kg'],
    'birth_length_cm': train_medians['birth_length_cm']
})

print("\nMAL-ED Missing after imputation (Train):", maled_train.isnull().sum().sum())
print("MAL-ED Missing after imputation (Test):", maled_test.isnull().sum().sum())

MAL-ED Missing before imputation (Train): 1869

Calculated Training Medians:
MUAC_mm            131.30
birth_weight_kg      3.06
birth_length_cm     48.70
dtype: float64

MAL-ED Missing after imputation (Train): 0
MAL-ED Missing after imputation (Test): 0


## Scale Numerical Features

In [14]:
# ---------------------------------------------------------
# Step 14: Scale Numerical Features (Fit on Train)
# ---------------------------------------------------------
from sklearn.preprocessing import StandardScaler

# --- DHS Scaling ---
# Isolate continuous columns (ignoring identifiers, targets, and one-hot encoded booleans)
dhs_num_cols = ['age_in_days', 'weight_kg', 'height_cm', 'MUAC_mm', 
                'birth_weight_kg', 'birth_length_cm', 'vaccination_adherence_score', 
                'missed_clinic_count_last_6_months', 'WHZ', 'WAZ', 'HAZ']

dhs_scaler = StandardScaler()
dhs_train[dhs_num_cols] = dhs_scaler.fit_transform(dhs_train[dhs_num_cols])
dhs_test[dhs_num_cols] = dhs_scaler.transform(dhs_test[dhs_num_cols])

print("DHS numerical features scaled successfully.")

# --- MAL-ED Scaling ---
maled_num_cols = ['age_in_days', 'weight_kg', 'height_cm', 'MUAC_mm', 
                  'birth_weight_kg', 'birth_length_cm', 'days_since_last_visit', 
                  'weight_delta_kg', 'WHZ', 'whz_velocity']

maled_scaler = StandardScaler()
maled_train[maled_num_cols] = maled_scaler.fit_transform(maled_train[maled_num_cols])
maled_test[maled_num_cols] = maled_scaler.transform(maled_test[maled_num_cols])

print("MAL-ED numerical features scaled successfully.")

DHS numerical features scaled successfully.
MAL-ED numerical features scaled successfully.


## SMOTE (DHS) & Class Weighting (MAL-ED

In [15]:
# %pip install imbalanced-learn

In [16]:
# ---------------------------------------------------------
# Step 15: SMOTE (DHS) and Class Weights (MAL-ED)
# ---------------------------------------------------------
from imblearn.over_sampling import SMOTE
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

# --- DHS: Apply SMOTE to Training Data Only ---
print(f"DHS Train shape before SMOTE: {dhs_train.shape}")

# Drop identifiers, targets, AND the raw datetime columns
cols_to_drop = ['child_id', 'label', 'date_of_birth', 'date_of_measurement']

X_dhs_train = dhs_train.drop(columns=[col for col in cols_to_drop if col in dhs_train.columns])
y_dhs_train = dhs_train['label']

X_dhs_test = dhs_test.drop(columns=[col for col in cols_to_drop if col in dhs_test.columns])
y_dhs_test = dhs_test['label']

# Apply SMOTE 
smote = SMOTE(random_state=42)
X_dhs_train_smoted, y_dhs_train_smoted = smote.fit_resample(X_dhs_train, y_dhs_train)  # type: ignore

print(f"DHS Train features shape after SMOTE: {X_dhs_train_smoted.shape}")
print("\nDHS Target distribution after SMOTE:")
print(y_dhs_train_smoted.value_counts())

# --- MAL-ED: Calculate Class Weights for LSTM ---
# (We don't need to drop dates from MAL-ED right now because class_weighting 
# only looks at the label column, not the features. We will handle MAL-ED 
# dates when we build the sequences).
classes = np.unique(maled_train['label'])
weights = compute_class_weight(class_weight='balanced', classes=classes, y=maled_train['label'])
maled_class_weights = dict(zip(classes, weights))

print("\nMAL-ED Computed Class Weights for LSTM (0=Normal, 1=MAM, 2=SAM):")
print(maled_class_weights)

DHS Train shape before SMOTE: (3996, 22)
DHS Train features shape after SMOTE: (9240, 18)

DHS Target distribution after SMOTE:
label
0    3080
2    3080
1    3080
Name: count, dtype: int64

MAL-ED Computed Class Weights for LSTM (0=Normal, 1=MAM, 2=SAM):
{np.int64(0): np.float64(0.599057714958775), np.int64(1): np.float64(1.6953333333333334), np.int64(2): np.float64(1.349787685774947)}


## Build Padded Sequences for LSTM

In [17]:
# %pip install tensorflow

In [18]:
# ---------------------------------------------------------
# Step 15: Build Padded Sequences for LSTM Input
# ---------------------------------------------------------
from tensorflow.keras.preprocessing.sequence import pad_sequences
import numpy as np

# 1. Define architecture branches based on your design
seq_features = ['age_in_days', 'weight_kg', 'height_cm', 'MUAC_mm', 
                'days_since_last_visit', 'weight_delta_kg', 'WHZ', 'whz_velocity']
static_features = ['sex', 'birth_weight_kg', 'birth_length_cm'] 
target_col = 'label'

def build_lstm_tensors(df, seq_cols, static_cols, target_col, max_len=None):
    """Converts longitudinal dataframe into padded 3D tensors."""
    sequences, statics, targets = [], [], []
    
    # Group by child to isolate individual timelines
    grouped = df.groupby('child_id')
    
    for _, group in grouped:
        # Ensure strict chronological order
        group = group.sort_values('visit_number')
        
        # Extract dynamic timeline (sequence)
        sequences.append(group[seq_cols].values)
        
        # Extract static traits (takes the first row since they don't change)
        statics.append(group[static_cols].iloc[0].values)
        
        # Extract timeline of labels
        targets.append(group[target_col].values)
        
    # 2. Pad sequences to uniform length
    # Using -999.0 as a highly distinct masking value for scaled clinical data
    X_seq_padded = pad_sequences(sequences, maxlen=max_len, padding='post', dtype='float32', value=-999.0)
    
    # Pad targets (using -1 as dummy padding for categorical labels)
    y_padded = pad_sequences(targets, maxlen=max_len, padding='post', dtype='int32', value=-1)
    
    return X_seq_padded, np.array(statics), y_padded

# 3. Apply to Training Set
X_maled_train_seq, X_maled_train_static, y_maled_train_seq = build_lstm_tensors(
    maled_train, seq_features, static_features, target_col
)

# 4. Apply to Test Set (force max_len to match the training set shape!)
max_time_steps = X_maled_train_seq.shape[1]
X_maled_test_seq, X_maled_test_static, y_maled_test_seq = build_lstm_tensors(
    maled_test, seq_features, static_features, target_col, max_len=max_time_steps
)

print(f"MAL-ED Train Tensors:")
print(f"  Sequential: {X_maled_train_seq.shape} | Static: {X_maled_train_static.shape} | Targets: {y_maled_train_seq.shape}")
print(f"MAL-ED Test Tensors:")
print(f"  Sequential: {X_maled_test_seq.shape} | Static: {X_maled_test_static.shape} | Targets: {y_maled_test_seq.shape}")

MAL-ED Train Tensors:
  Sequential: (560, 6, 8) | Static: (560, 3) | Targets: (560, 6)
MAL-ED Test Tensors:
  Sequential: (140, 6, 8) | Static: (140, 3) | Targets: (140, 6)


## Export Prepared Data

In [19]:
# ---------------------------------------------------------
# Step 16: Export Datasets for Model Training (Notebook 2)
# ---------------------------------------------------------
import os

# Create a designated directory for clean data
output_dir = "../data/prepared_data"
os.makedirs(output_dir, exist_ok=True)

# --- 1. Export DHS (Shallow Learning) Data ---
# Since SMOTE retains pandas DataFrames, standard CSVs are perfect here.
X_dhs_train_smoted.to_csv(f"{output_dir}/X_dhs_train.csv", index=False)
y_dhs_train_smoted.to_csv(f"{output_dir}/y_dhs_train.csv", index=False)
X_dhs_test.to_csv(f"{output_dir}/X_dhs_test.csv", index=False)
y_dhs_test.to_csv(f"{output_dir}/y_dhs_test.csv", index=False)

print("DHS datasets exported successfully.")

# --- 2. Export MAL-ED (Deep Learning) Tensors ---
# Numpy compressed arrays (.npz) are ideal for 3D tensors.
np.savez_compressed(
    f"{output_dir}/maled_tensors.npz",
    X_train_seq=X_maled_train_seq,
    X_train_static=X_maled_train_static,
    y_train_seq=y_maled_train_seq,
    X_test_seq=X_maled_test_seq,
    X_test_static=X_maled_test_static,
    y_test_seq=y_maled_test_seq,
    class_weights=maled_class_weights # Saving the weights for Keras fit!
)

print("MAL-ED 3D tensors exported successfully.")
print("\n✅ Notebook 1: Data Preparation is 100% Complete.")

DHS datasets exported successfully.
MAL-ED 3D tensors exported successfully.

✅ Notebook 1: Data Preparation is 100% Complete.


In [20]:
# ---------------------------------------------------------
# Final Export: Preprocessing Artifacts for Django
# ---------------------------------------------------------
import joblib
import os

# Ensure the models directory exists
os.makedirs("../models", exist_ok=True)

# Export the fitted MICE imputer and the specific MAL-ED StandardScaler
joblib.dump(mice_imputer, '../models/bluecradle_imputer.pkl')
joblib.dump(maled_scaler, '../models/bluecradle_scaler.pkl')

print("✅ MICE Imputer and MAL-ED Scaler successfully exported to models/ directory.")

✅ MICE Imputer and MAL-ED Scaler successfully exported to models/ directory.
